# File Formats

I present three data formats, feather, parquet and hdf but it exists several more like [Apache Avro](http://avro.apache.org/docs/current/) or [Apache ORC](https://orc.apache.org).

These data formats may be more appropriate in certain situations.
However, the software needed to handle them is either more difficult
to install, incomplete, or more difficult to use because less
documentation is provided. For ORC and AVRO the python libraries
offered are less well maintained than the formats we will see. You can find many on
the web but it is hard to know which one is the most stable.
- [pyorc](https://github.com/noirello/pyorc)
- [avro](https://avro.apache.org/docs/1.10.0/gettingstartedpython.html) and [fastavro](https://github.com/fastavro/fastavro)
The following formats are supported
by pandas and apache arrow. These softwares are supported by very strong communities.

## Feather

For light data, it is recommanded to use [Feather](https://github.com/wesm/feather). It is a fast, interoperable data frame storage that comes with bindings for python and R.

Feather uses also the Apache Arrow columnar memory specification to represent binary data on disk. This makes read and write operations very fast.

## Parquet file format

[Parquet format](https://github.com/apache/parquet-format) is a common binary data store, used particularly in the Hadoop/big-data sphere. It provides several advantages relevant to big-data processing:

The Apache Parquet project provides a standardized open-source columnar storage format for use in data analysis systems. It was created originally for use in Apache Hadoop with systems like Apache Drill, Apache Hive, Apache Impala, and Apache Spark adopting it as a shared standard for high performance data IO.



## Hierarchical Data Format

 [HDF](https://en.wikipedia.org/wiki/Hierarchical_Data_Format) is a self-describing data format
allowing an application to interpret the structure and
contents of a file with no outside information.
One HDF file can hold a mix of related objects
which can be accessed as a group or as individual objects.

Let's create some big dataframe with consitent data (Floats) and 10% of missing values:

In [1]:
!pip install pyarrow pandas lorem

from pyarrow import feather
import pandas as pd
import numpy as np
arr = np.random.randn(500000) # 10% nulls
arr[::10] = np.nan
df = pd.DataFrame({'column_{0}'.format(i): arr for i in range(10)})

In [2]:
%time df.to_csv('test.csv')

CPU times: user 9.77 s, sys: 138 ms, total: 9.91 s
Wall time: 12.3 s


In [3]:
!rm -f test.h5

In [4]:
%time df.to_hdf("test.h5", key="test")

CPU times: user 206 ms, sys: 73.2 ms, total: 279 ms
Wall time: 477 ms


In [5]:
%time df.to_parquet('test.parquet')

CPU times: user 307 ms, sys: 49.5 ms, total: 357 ms
Wall time: 411 ms


In [6]:
%time df.to_feather('test.feather')

CPU times: user 187 ms, sys: 25.9 ms, total: 213 ms
Wall time: 223 ms


In [7]:
!du -sh test.*

88M	test.csv
36M	test.feather
42M	test.h5
38M	test.parquet


In [8]:
%%time
df = pd.read_csv("test.csv")
len(df)

CPU times: user 1.21 s, sys: 78.5 ms, total: 1.29 s
Wall time: 1.69 s


500000

In [9]:
%%time
df = pd.read_hdf("test.h5")
len(df)

CPU times: user 34.6 ms, sys: 28.8 ms, total: 63.4 ms
Wall time: 84.1 ms


500000

In [10]:
%%time
df = pd.read_parquet("test.parquet")
len(df)

CPU times: user 76.7 ms, sys: 79.3 ms, total: 156 ms
Wall time: 271 ms


500000

In [11]:
%%time
df = pd.read_feather("test.feather")
len(df)

CPU times: user 33.5 ms, sys: 34.3 ms, total: 67.8 ms
Wall time: 82 ms


500000

In [12]:
# Now we create a new big dataframe with a column of strings

In [13]:
import numpy as np
import pandas as pd
from lorem import sentence

words = np.array(sentence().strip().lower().replace(".", " ").split())

# Set the seed so that the numbers can be reproduced.
np.random.seed(0)
n = 1000000
df = pd.DataFrame(np.c_[np.random.randn(n, 5),
                  np.random.randint(0,10,(n, 2)),
                  np.random.randint(0,1,(n, 2)),
np.array([np.random.choice(words) for i in range(n)])] ,
columns=list('ABCDEFGHIJ'))

df["A"][::10] = np.nan
len(df)


/tmp/ipykernel_2270/3509618284.py:16: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["A"][::10] = np.nan


1000000

In [14]:
%%time
df.to_csv('test.csv', index=False)

CPU times: user 5.67 s, sys: 98.7 ms, total: 5.77 s
Wall time: 5.85 s


In [15]:
%%time
df.to_hdf('test.h5', key="test", mode="w")

<timed eval>:1: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'], dtype='object')]



CPU times: user 2.52 s, sys: 636 ms, total: 3.15 s
Wall time: 3.2 s


In [16]:
%%time
df.to_feather('test.feather')

CPU times: user 1.08 s, sys: 203 ms, total: 1.29 s
Wall time: 1.12 s


In [17]:
%%time
df.to_parquet('test.parquet')

CPU times: user 1.53 s, sys: 128 ms, total: 1.66 s
Wall time: 1.67 s


In [18]:
%%time
df = pd.read_csv("test.csv")
len(df)

CPU times: user 1.71 s, sys: 145 ms, total: 1.85 s
Wall time: 1.92 s


1000000

In [19]:
%%time
df = pd.read_hdf("test.h5")
len(df)

CPU times: user 921 ms, sys: 787 ms, total: 1.71 s
Wall time: 1.74 s


1000000

In [20]:
%%time
df = pd.read_feather('test.feather')
len(df)

CPU times: user 1.28 s, sys: 373 ms, total: 1.65 s
Wall time: 1.6 s


1000000

In [21]:
%%time
df = pd.read_parquet('test.parquet')
len(df)


CPU times: user 1.87 s, sys: 426 ms, total: 2.29 s
Wall time: 1.88 s


1000000

In [22]:
df.head(10)

,A,B,C,D,E,F,G,H,I,J
0,None,0.4001572083672233,0.9787379841057392,2.240893199201458,1.8675579901499675,0,4,0,0,ut
1,-0.977277879876411,0.9500884175255894,-0.1513572082976979,-0.10321885179355784,0.41059850193837233,5,5,0,0,ut
2,0.144043571160878,1.454273506962975,0.7610377251469934,0.12167501649282841,0.44386323274542566,6,1,0,0,etincidunt
3,0.33367432737426683,1.4940790731576061,-0.20515826376580087,0.31306770165090136,-0.8540957393017248,0,5,0,0,dolore
4,-2.5529898158340787,0.6536185954403606,0.8644361988595057,-0.7421650204064419,2.2697546239876076,6,7,0,0,ut
5,-1.4543656745987648,0.04575851730144607,-0.1871838500258336,1.5327792143584575,1.469358769900285,6,0,0,0,ut
6,0.1549474256969163,0.37816251960217356,-0.8877857476301128,-1.980796468223927,-0.3479121493261526,8,0,0,0,ut
7,0.15634896910398005,1.2302906807277207,1.2023798487844113,-0.3873268174079523,-0.30230275057533557,5,5,0,0,ut
8,-1.0485529650670926,-1.4200179371789752,-1.7062701906250126,1.9507753952317897,-0.5096521817516535,7,5,0,0,etincidunt
9,-0.4380743016111864,-1.2527953600499262,0.7774903558319101,-1.6138978475579515,-0.2127402802139687,2,0,0,0,ut


In [23]:
df['J'] = pd.Categorical(df.J)

In [24]:
%time df.to_feather('test.feather')


CPU times: user 800 ms, sys: 147 ms, total: 947 ms
Wall time: 811 ms


In [25]:
%time df.to_parquet('test.parquet')

CPU times: user 1.26 s, sys: 121 ms, total: 1.38 s
Wall time: 1.37 s


In [26]:
%%time
df = pd.read_feather('test.feather')
len(df)

CPU times: user 1.09 s, sys: 333 ms, total: 1.42 s
Wall time: 1.39 s


1000000

In [27]:
%%time
df = pd.read_parquet('test.parquet')
len(df)

CPU times: user 1.88 s, sys: 411 ms, total: 2.29 s
Wall time: 1.87 s


1000000

## Feather or Parquet

- Parquet format is designed for long-term storage, where Arrow is more intended for short term or ephemeral storage because files volume are larger.
- Parquet is usually more expensive to write than Feather as it features more layers of encoding and compression.
- Feather is unmodified raw columnar Arrow memory. We will probably add simple compression to Feather in the future.
- Due to dictionary encoding, RLE encoding, and data page compression, Parquet files will often be much smaller than Feather files
- Parquet is a standard storage format for analytics that's supported by Spark. So if you are doing analytics, Parquet is a good option as a reference storage format for query by multiple systems

[source stackoverflow](https://stackoverflow.com/questions/48083405/what-are-the-differences-between-feather-and-parquet)

## Apache Arrow

[Arrow](https://arrow.apache.org/docs/python/) is a columnar in-memory analytics layer designed to accelerate big data.
It houses a set of canonical in-memory representations of
hierarchical data along with multiple language-bindings
for structure manipulation. Arrow offers an unified way to be able to
share the same data representation among languages and it will certainly be
the next standard to store dataframes in all languages.

- [R package](https://cran.r-project.org/web/packages/arrow/index.html)
- [Julia package](https://github.com/JuliaData/Arrow.jl)
- [GitHub project](https://github.com/apache/arrow)

![](https://github.com/pnavaro/big-data/blob/master/notebooks/images/arrow_ecosystem.png?raw=1)

Apache Arrow is an ideal in-memory transport layer for data that is being read or written with Parquet files. [PyArrow](https://arrow.apache.org/docs/python/) includes Python bindings to read and write Parquet files with pandas.

- columnar storage, only read the data of interest
- efficient binary packing
- choice of compression algorithms and encoding
- split data into files, allowing for parallel processing
- range of logical types
- statistics stored in metadata allow for skipping unneeded chunks
- data partitioning using the directory structure

![arrow](https://github.com/pnavaro/big-data/blob/master/notebooks/images/arrow.png?raw=1)

- https://arrow.apache.org/docs/python/csv.html
- https://arrow.apache.org/docs/python/feather.html
- https://arrow.apache.org/docs/python/parquet.html

Example:

```py
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
arr = np.random.randn(500000) # 10% nulls
arr[::10] = np.nan
df = pd.DataFrame({'column_{0}'.format(i): arr for i in range(10)})

hdfs = pa.hdfs.connect()
table = pa.Table.from_pandas(df)
pq.write_to_dataset(table, root_path="test", filesystem=hdfs)
hdfs.ls("test")

```

### Read CSV from HDFS

Put the file test.csv on hdfs system

```python
from pyarrow import csv
with hdfs.open("/data/nycflights/1999.csv", "rb") as f:
 df = pd.read_csv(f, nrows = 10)
print(df.head())
```

### Read Parquet File from HDFS with pandas

```python
import pandas as pd
wikipedia = pd.read_parquet("hdfs://svmass2.mass.uhb.fr:54310/data/pagecounts-parquet/part-00007-8575060f-6b57-45ea-bf1d-cd77b6141f70.snappy.parquet", engine=’pyarrow’)
print(wikipedia.head())
```
### Read Parquet File with pyarrow

```py
table = pq.read_table("example.parquet")
```

### Writing a parquet file from Apache Arrow
```py
pq.write_table(table, "example.parquet")
```

### Check metadata
```py
parquet_file = pq.ParquetFile("example.parquet")
print(parquet_file.metadata)
```

### See schema
```py
print(parquet_file.schema)
```

### Connect to the Hadoop file system

```py
hdfs = pa.hdfs.connect()

# copy to local
with hdfs.open("user.txt", "rb") as f:
    f.download("user.text")

# write parquet file on hdfs
with open("example.parquet", "rb") as f:
    pa.HadoopFileSystem.upload(hdfs, "example.parquet", f)

# List files
for f in hdfs.ls("/user/navaro_p"):
    print(f)

# create a small dataframe and write it to hadoop file system
small_df = pd.DataFrame(np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]]), columns=['a', 'b', 'c'])
table = pa.Table.from_pandas(small_df)
pq.write_table(table, "small_df.parquet", filesystem=hdfs)


# Read files from Hadoop with pandas
with hdfs.open("/data/irmar.csv") as f:
    df = pd.read_csv(f)

print(df.head())

# Read parquet file from Hadoop with pandas
server = "hdfs://svmass2.mass.uhb.fr:54310"
path = "data/pagecounts-parquet/part-00007-8575060f-6b57-45ea-bf1d-cd77b6141f70.snappy.parquet"
pagecount = pd.read_parquet(os.path.join(server, path), engine="pyarrow")
print(pagecount.head())

# Read parquet file from Hadoop with pyarrow
table = pq.read_table(os.path.join(server,path))
print(table.schema)
df = table.to_pandas()
print(df.head())
```

### Exercise

- Take the second dataframe with string as last column
- Create an arrow table from pandas dataframe
- Write the file test.parquet from arrow table
- Print metadata from this parquet file
- Print schema
- Upload the file to hadoop file system
- Read this file from hadoop file system and print dataframe head


Hint: check the doc https://arrow.apache.org/docs/python/parquet.html

## Jawaban Revisi Exercise — PyArrow Parquet dan Hadoop File System

Bagian ini menjawab semua instruksi exercise secara berurutan:

1. Mengambil dataframe kedua dengan kolom terakhir bertipe string.
2. Membuat Arrow Table dari pandas dataframe.
3. Menulis file `test.parquet` dari Arrow Table.
4. Menampilkan metadata file Parquet.
5. Menampilkan schema.
6. Upload file ke Hadoop File System.
7. Membaca file dari Hadoop File System dan menampilkan dataframe `head()`.

Catatan: bagian HDFS membutuhkan environment yang sudah memiliki Hadoop/HDFS aktif.


### 1. Import library

Library utama yang digunakan adalah `pandas`, `pyarrow`, dan `pyarrow.parquet`.


In [28]:
# Install library jika belum tersedia
!pip -q install pandas pyarrow lorem

import os
import shutil
import subprocess
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

print("Pandas version:", pd.__version__)
print("PyArrow version:", pa.__version__)


Pandas version: 2.2.2
PyArrow version: 18.1.0


### 2. Take the second dataframe with string as last column

Pada notebook ini, dataframe kedua adalah dataframe besar yang memiliki kolom `A` sampai `J`, dengan kolom terakhir `J` berisi data string.  
Cell ini dibuat aman: jika dataframe belum tersedia karena runtime di-reset, maka dataframe kedua dibuat ulang dengan struktur yang sama.


In [29]:
# Take the second dataframe with string as last column

# Jika df dari bagian sebelumnya belum tersedia, buat ulang dataframe kedua.
# Struktur mengikuti dataframe kedua pada notebook: kolom A-I numerik dan kolom J string.
try:
    df
    df_second = df.copy()
except NameError:
    from lorem import sentence

    words = np.array(sentence().strip().lower().replace(".", " ").split())

    np.random.seed(0)
    n = 100_000

    df_second = pd.DataFrame(
        np.c_[
            np.random.randn(n, 5),
            np.random.randint(0, 10, (n, 2)),
            np.random.randint(0, 2, (n, 2)),
            np.array([np.random.choice(words) for _ in range(n)])
        ],
        columns=list("ABCDEFGHIJ")
    )

    df_second.loc[::10, "A"] = np.nan

# Pastikan kolom J menjadi kolom terakhir.
if "J" in df_second.columns and df_second.columns[-1] != "J":
    other_cols = [col for col in df_second.columns if col != "J"]
    df_second = df_second[other_cols + ["J"]]

# Pastikan kolom terakhir bertipe string.
last_col = df_second.columns[-1]
df_second[last_col] = df_second[last_col].astype(str)

print("Nama kolom terakhir:", last_col)
print("Tipe data kolom terakhir:", df_second[last_col].dtype)
print("Jumlah baris:", len(df_second))
print("Jumlah kolom:", len(df_second.columns))

df_second.head()


Nama kolom terakhir: J
Tipe data kolom terakhir: object
Jumlah baris: 1000000
Jumlah kolom: 10


,A,B,C,D,E,F,G,H,I,J
0,None,0.4001572083672233,0.9787379841057392,2.240893199201458,1.8675579901499675,0,4,0,0,ut
1,-0.977277879876411,0.9500884175255894,-0.1513572082976979,-0.10321885179355784,0.41059850193837233,5,5,0,0,ut
2,0.144043571160878,1.454273506962975,0.7610377251469934,0.12167501649282841,0.44386323274542566,6,1,0,0,etincidunt
3,0.33367432737426683,1.4940790731576061,-0.20515826376580087,0.31306770165090136,-0.8540957393017248,0,5,0,0,dolore
4,-2.5529898158340787,0.6536185954403606,0.8644361988595057,-0.7421650204064419,2.2697546239876076,6,7,0,0,ut


### 3. Create an Arrow Table from pandas dataframe

Dataframe kedua diubah menjadi Arrow Table menggunakan `pa.Table.from_pandas()`.


In [30]:
# Create an Arrow table from pandas dataframe
arrow_table = pa.Table.from_pandas(df_second, preserve_index=False)

print("Arrow Table berhasil dibuat.")
print("Jumlah baris:", arrow_table.num_rows)
print("Jumlah kolom:", arrow_table.num_columns)

print("\nArrow Schema:")
print(arrow_table.schema)


Arrow Table berhasil dibuat.
Jumlah baris: 1000000
Jumlah kolom: 10

Arrow Schema:
A: string
B: string
C: string
D: string
E: string
F: string
G: string
H: string
I: string
J: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1095


### 4. Write the file `test.parquet` from Arrow Table

File yang diminta pada soal harus bernama `test.parquet`.


In [31]:
# Write the file test.parquet from Arrow table
local_parquet_file = "test.parquet"

pq.write_table(
    arrow_table,
    local_parquet_file,
    compression="snappy"
)

print("File berhasil dibuat:", local_parquet_file)
print("Ukuran file:", round(os.path.getsize(local_parquet_file) / (1024 ** 2), 4), "MB")


File berhasil dibuat: test.parquet
Ukuran file: 79.0578 MB


### 5. Print metadata from this parquet file

Metadata menunjukkan informasi file Parquet, seperti jumlah baris, jumlah row group, format, dan informasi kolom.


In [32]:
# Print metadata from this parquet file
metadata = pq.read_metadata(local_parquet_file)

print("=== PARQUET METADATA ===")
print(metadata)


=== PARQUET METADATA ===
  created_by: parquet-cpp-arrow version 18.1.0
  num_columns: 10
  num_rows: 1000000
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 4607


### 6. Print schema

Schema menunjukkan struktur kolom dan tipe data yang tersimpan di file Parquet.


In [33]:
# Print schema
parquet_file = pq.ParquetFile(local_parquet_file)

print("=== PARQUET SCHEMA ===")
print(parquet_file.schema)


=== PARQUET SCHEMA ===
required group field_id=-1 schema {
  optional binary field_id=-1 A (String);
  optional binary field_id=-1 B (String);
  optional binary field_id=-1 C (String);
  optional binary field_id=-1 D (String);
  optional binary field_id=-1 E (String);
  optional binary field_id=-1 F (String);
  optional binary field_id=-1 G (String);
  optional binary field_id=-1 H (String);
  optional binary field_id=-1 I (String);
  optional binary field_id=-1 J (String);
}



### Review penyebab `hdfs` tidak ditemukan

Pesan `Perintah 'hdfs' tidak ditemukan` muncul karena environment yang dipakai, misalnya Google Colab, **tidak menyediakan Hadoop/HDFS secara default**.  
Library `pyarrow` hanya digunakan untuk membaca/menulis file Parquet, tetapi **tidak otomatis menginstall command Hadoop** seperti `hdfs dfs`.

Perintah yang benar untuk mengecek HDFS:

- Di Linux/Colab: `!which hdfs`
- Di Windows PowerShell: `where hdfs`

Jika `which hdfs` kosong, berarti Hadoop CLI belum tersedia di runtime. Oleh karena itu, bagian berikut menambahkan setup Hadoop/HDFS agar perintah `hdfs dfs` bisa digunakan di Google Colab atau Linux runtime.


### 7. Setup Hadoop/HDFS untuk Google Colab atau Linux Runtime

Jalankan cell ini sebelum upload file ke HDFS.  
Cell ini akan:

1. Mengecek apakah command `hdfs` sudah ada.
2. Jika belum ada, menginstall Java dan Hadoop.
3. Mengatur `JAVA_HOME`, `HADOOP_HOME`, dan `PATH`.
4. Membuat konfigurasi HDFS pseudo-distributed.
5. Menjalankan NameNode dan DataNode.
6. Mengecek apakah HDFS sudah aktif.

Catatan: cell ini membutuhkan internet karena harus mengunduh Hadoop jika belum tersedia.


In [34]:
# Setup Hadoop/HDFS for Google Colab or Linux runtime
import os
import shutil
import subprocess
import time
from pathlib import Path

def run_cmd(cmd, check=True, show_output=True):
    """
    Helper untuk menjalankan command shell.
    """
    print("\n$ " + cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    if show_output and result.stdout:
        print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command gagal dengan exit code {result.returncode}: {cmd}")
    return result

print("Cek awal command hdfs:", shutil.which("hdfs"))

# Konfigurasi dasar
HADOOP_VERSION = "3.3.6"
HADOOP_HOME = f"/content/hadoop-{HADOOP_VERSION}"
HADOOP_TAR = f"/content/hadoop-{HADOOP_VERSION}.tar.gz"
JAVA_HOME = "/usr/lib/jvm/java-11-openjdk-amd64"

# Jika hdfs belum ditemukan, install Java dan Hadoop.
if shutil.which("hdfs") is None:
    print("Command hdfs belum ditemukan. Menyiapkan Hadoop untuk runtime ini...")

    # Install Java dan utilitas pendukung
    run_cmd("apt-get update -qq", check=True, show_output=False)
    run_cmd("apt-get install -y -qq openjdk-11-jdk-headless wget rsync", check=True, show_output=True)

    # Download Hadoop jika folder belum ada
    if not Path(HADOOP_HOME).exists():
        if not Path(HADOOP_TAR).exists():
            run_cmd(
                f"wget -q https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VERSION}/hadoop-{HADOOP_VERSION}.tar.gz -O {HADOOP_TAR}",
                check=True,
                show_output=True
            )

        run_cmd(f"tar -xzf {HADOOP_TAR} -C /content", check=True, show_output=True)

# Set environment variables agar hdfs bisa ditemukan oleh Python dan shell command.
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["HADOOP_CONF_DIR"] = f"{HADOOP_HOME}/etc/hadoop"
os.environ["PATH"] = (
    f"{HADOOP_HOME}/bin:"
    f"{HADOOP_HOME}/sbin:"
    + os.environ.get("PATH", "")
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("hdfs path:", shutil.which("hdfs"))

# Pastikan hdfs sudah tersedia
if shutil.which("hdfs") is None:
    raise EnvironmentError("HDFS masih belum ditemukan. Periksa koneksi internet atau proses instalasi Hadoop.")

# Tulis konfigurasi Hadoop/HDFS
conf_dir = Path(os.environ["HADOOP_CONF_DIR"])
conf_dir.mkdir(parents=True, exist_ok=True)

hadoop_env = conf_dir / "hadoop-env.sh"
hadoop_env_text = hadoop_env.read_text() if hadoop_env.exists() else ""
if "export JAVA_HOME=" not in hadoop_env_text:
    with hadoop_env.open("a") as f:
        f.write(f"\nexport JAVA_HOME={JAVA_HOME}\n")

core_site = """<?xml version="1.0"?>
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
</configuration>
"""

hdfs_site = """<?xml version="1.0"?>
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>file:///content/hadoop_tmp/hdfs/namenode</value>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>file:///content/hadoop_tmp/hdfs/datanode</value>
  </property>
</configuration>
"""

(conf_dir / "core-site.xml").write_text(core_site)
(conf_dir / "hdfs-site.xml").write_text(hdfs_site)

# Buat folder data HDFS
Path("/content/hadoop_tmp/hdfs/namenode").mkdir(parents=True, exist_ok=True)
Path("/content/hadoop_tmp/hdfs/datanode").mkdir(parents=True, exist_ok=True)

# Format NameNode hanya jika belum pernah diformat
namenode_current = Path("/content/hadoop_tmp/hdfs/namenode/current")
if not namenode_current.exists():
    run_cmd("hdfs namenode -format -force -nonInteractive", check=True, show_output=False)
else:
    print("NameNode sudah pernah diformat, lanjut menjalankan daemon.")

# Stop daemon lama jika ada, lalu start NameNode dan DataNode.
run_cmd("hdfs --daemon stop datanode", check=False, show_output=False)
run_cmd("hdfs --daemon stop namenode", check=False, show_output=False)
time.sleep(2)

run_cmd("hdfs --daemon start namenode", check=True, show_output=True)
run_cmd("hdfs --daemon start datanode", check=True, show_output=True)
time.sleep(5)

# Cek HDFS aktif
run_cmd("jps", check=False, show_output=True)
run_cmd("hdfs dfs -ls /", check=False, show_output=True)

print("Setup Hadoop/HDFS selesai. Command hdfs sudah siap digunakan.")


Cek awal command hdfs: None
Command hdfs belum ditemukan. Menyiapkan Hadoop untuk runtime ini...

$ apt-get update -qq

$ apt-get install -y -qq openjdk-11-jdk-headless wget rsync
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../rsync_3.2.7-0ubuntu0.22.04.7_amd64.deb ...
Unpacking rsync (3.2.7-0ubuntu0.22.04.7) over (3.2.7-0ubuntu0.22.04.6) ...
Selecting previously unselected package openjdk-11-jre-headles

### 8. Upload the file `test.parquet` to Hadoop File System

Bagian ini mengupload file `test.parquet` ke HDFS menggunakan `hdfs dfs -put`.

Path HDFS yang digunakan:

`/user/nela/parquet_exercise/test.parquet`


In [35]:
# Upload the file to Hadoop file system
import shutil
import subprocess

hdfs_dir = "/user/nela/parquet_exercise"
hdfs_file = f"{hdfs_dir}/test.parquet"

if shutil.which("hdfs") is None:
    raise EnvironmentError(
        "Command hdfs belum ditemukan. Jalankan cell Setup Hadoop/HDFS terlebih dahulu."
    )

if not Path(local_parquet_file).exists():
    raise FileNotFoundError(
        "File test.parquet belum ada. Jalankan cell pembuatan file Parquet terlebih dahulu."
    )

# Buat folder HDFS dan upload file
run_cmd(f"hdfs dfs -mkdir -p {hdfs_dir}", check=True, show_output=True)
run_cmd(f"hdfs dfs -put -f {local_parquet_file} {hdfs_file}", check=True, show_output=True)

# Cek file di HDFS
print("File di HDFS:")
run_cmd(f"hdfs dfs -ls {hdfs_dir}", check=True, show_output=True)



$ hdfs dfs -mkdir -p /user/nela/parquet_exercise

$ hdfs dfs -put -f test.parquet /user/nela/parquet_exercise/test.parquet
File di HDFS:

$ hdfs dfs -ls /user/nela/parquet_exercise
Found 1 items
-rw-r--r--   1 root supergroup   82898133 2026-06-27 17:30 /user/nela/parquet_exercise/test.parquet



CompletedProcess(args='hdfs dfs -ls /user/nela/parquet_exercise', returncode=0, stdout='Found 1 items\n-rw-r--r--   1 root supergroup   82898133 2026-06-27 17:30 /user/nela/parquet_exercise/test.parquet\n')

### 9. Read file from Hadoop File System and print dataframe head

PyArrow kadang tidak bisa membaca `hdfs://` secara langsung jika runtime tidak memiliki native HDFS connector.  
Agar tetap sesuai tugas, file dibaca dari HDFS menggunakan `hdfs dfs -get`, lalu isi file Parquet ditampilkan dengan pandas `head()`.

Alurnya:

`HDFS test.parquet` → `hdfs dfs -get` → `read_table()` → `to_pandas()` → `head()`


In [36]:
# Read this file from Hadoop file system and print dataframe head
temp_local_file = "test_from_hdfs.parquet"

if shutil.which("hdfs") is None:
    raise EnvironmentError(
        "Command hdfs belum ditemukan. Jalankan cell Setup Hadoop/HDFS terlebih dahulu."
    )

# Pastikan file ada di HDFS
run_cmd(f"hdfs dfs -test -e {hdfs_file}", check=True, show_output=False)

# Ambil file dari HDFS ke local temporary file
run_cmd(f"hdfs dfs -get -f {hdfs_file} {temp_local_file}", check=True, show_output=True)

# Baca file parquet yang berasal dari HDFS
table_from_hdfs = pq.read_table(temp_local_file)
df_from_hdfs = table_from_hdfs.to_pandas()

print("Dataframe hasil baca dari Hadoop File System:")
display(df_from_hdfs.head())



$ hdfs dfs -test -e /user/nela/parquet_exercise/test.parquet

$ hdfs dfs -get -f /user/nela/parquet_exercise/test.parquet test_from_hdfs.parquet
Dataframe hasil baca dari Hadoop File System:


,A,B,C,D,E,F,G,H,I,J
0,None,0.4001572083672233,0.9787379841057392,2.240893199201458,1.8675579901499675,0,4,0,0,ut
1,-0.977277879876411,0.9500884175255894,-0.1513572082976979,-0.10321885179355784,0.41059850193837233,5,5,0,0,ut
2,0.144043571160878,1.454273506962975,0.7610377251469934,0.12167501649282841,0.44386323274542566,6,1,0,0,etincidunt
3,0.33367432737426683,1.4940790731576061,-0.20515826376580087,0.31306770165090136,-0.8540957393017248,0,5,0,0,dolore
4,-2.5529898158340787,0.6536185954403606,0.8644361988595057,-0.7421650204064419,2.2697546239876076,6,7,0,0,ut


### 10. Optional: Coba baca langsung dari HDFS URI

Cell ini opsional. Jika native HDFS connector di runtime mendukung, PyArrow dapat membaca langsung dari URI `hdfs://localhost:9000/...`.  
Jika gagal, tidak masalah, karena cell sebelumnya sudah membaca file dari HDFS menggunakan `hdfs dfs -get`.


In [37]:
# Optional direct HDFS URI read
hdfs_uri = "hdfs://localhost:9000/user/nela/parquet_exercise/test.parquet"

try:
    table_direct_hdfs = pq.read_table(hdfs_uri)
    df_direct_hdfs = table_direct_hdfs.to_pandas()

    print("Berhasil membaca langsung dari HDFS URI:")
    display(df_direct_hdfs.head())

except Exception as e:
    print("Read langsung dari HDFS URI belum didukung di runtime ini.")
    print("Ini tidak masalah karena file sudah berhasil dibaca dari HDFS menggunakan hdfs dfs -get.")
    print("Detail error:", e)


Read langsung dari HDFS URI belum didukung di runtime ini.
Ini tidak masalah karena file sudah berhasil dibaca dari HDFS menggunakan hdfs dfs -get.
Detail error: HDFS connection failed


### 11. Pengecekan lokal tambahan

Cell ini bukan pengganti HDFS.  
Cell ini hanya untuk memastikan file `test.parquet` yang dibuat dari Arrow Table bisa dibaca dengan benar secara lokal.


In [38]:
# Local check only
local_read_table = pq.read_table(local_parquet_file)
local_read_df = local_read_table.to_pandas()

print("Dataframe hasil baca lokal dari test.parquet:")
display(local_read_df.head())


Dataframe hasil baca lokal dari test.parquet:


,A,B,C,D,E,F,G,H,I,J
0,None,0.4001572083672233,0.9787379841057392,2.240893199201458,1.8675579901499675,0,4,0,0,ut
1,-0.977277879876411,0.9500884175255894,-0.1513572082976979,-0.10321885179355784,0.41059850193837233,5,5,0,0,ut
2,0.144043571160878,1.454273506962975,0.7610377251469934,0.12167501649282841,0.44386323274542566,6,1,0,0,etincidunt
3,0.33367432737426683,1.4940790731576061,-0.20515826376580087,0.31306770165090136,-0.8540957393017248,0,5,0,0,dolore
4,-2.5529898158340787,0.6536185954403606,0.8644361988595057,-0.7421650204064419,2.2697546239876076,6,7,0,0,ut
